# Create or append Virtual Icechunk Store from SHYFEM forecast NetCDF files

* This notebook appends Taranto SHYFEM forecast data to Icechunk store using date-based set logic.
* If no repo exists, a new one will be created with references to all the existing NetCDF files. 

**Append Methodology:**
1. **`set_repo`**: extract all dates currently present in the Icechunk store's `time` coordinate
2. **`set_cloud`**: Scan the S3 bucket for all available NOS files and extract their dates.
3. **`new_dates`**: Calculate the difference (`set_cloud - set_repo`) to determine exactly which days need to be written for creation or appended.


In [ ]:
import warnings
import os
import pandas as pd
import fsspec
import xarray as xr
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import icechunk
from obstore.store import from_url
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import S3Store

In [ ]:
# Load credentials
_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env', override=True)

# Configuration
storage_endpoint = os.environ['ENDPOINT_URL']
storage_bucket = 'protocoast-data'
storage_name = 'taranto-icechunk-tubitak-v1'
bucket_url = f"s3://{storage_bucket}"

# Setup Filesystem
fs = fsspec.filesystem('s3', anon=False, endpoint_url=storage_endpoint, 
                       skip_instance_cache=True, use_listings_cache=False)

In [ ]:
fs.ls(f'{storage_bucket}/icechunk')

In [ ]:
# Define Icechunk Storage & Config
storage = icechunk.s3_storage(
    bucket=storage_bucket,
    prefix=f"icechunk/{storage_name}",
    from_env=True,
    endpoint_url=storage_endpoint,
    region='not-used',
    force_path_style=True,
)

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f"{bucket_url}/",
        store=icechunk.s3_store(region="not-used", anonymous=False, s3_compatible=True, 
                                force_path_style=True, endpoint_url=storage_endpoint),
    ),
)

credentials = icechunk.containers_credentials({f"{bucket_url}/": icechunk.s3_credentials(anonymous=False)})

store_obj = S3Store(
    bucket="protocoast-data",
    endpoint=storage_endpoint, # e.g., "https://rustfs.vm.fedcloud.eu:9001"
    region="not-used",
)

registry = ObjectStoreRegistry({bucket_url: store_obj})
parser = HDFParser()

### Step 1: Create Date Sets (`set_repo` vs `set_cloud`)

In [ ]:
# --- 1. Get Dates from Icechunk Repo (set_repo) ---
try:
    repo = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=credentials)
    session = repo.readonly_session("main")
    ds = xr.open_zarr(session.store, consolidated=False, chunks={})
    
    if 'time' in ds.coords:
        # Extract dates as YYYYMMDD strings
        dates = pd.to_datetime(ds.time.values) + pd.Timedelta(days=1)
        set_repo = set(dates.strftime('%Y%m%d'))
    else:
        set_repo = set()
        
except Exception as e:
    print(f"Repo access failed or empty ({e}). Assuming set_repo is empty.")
    repo = None
    set_repo = set()

print(f"set_repo: {len(set_repo)} dates found.")

# --- 2. Get Dates from Cloud Bucket (set_cloud) ---
print("Scanning S3 for NOS files...")
nos_files = fs.glob(f'{bucket_url}/full_dataset/shyfem/taranto/forecast/*/*nos*.nc')

set_cloud = set()
date_to_files_map = {} # Helper to quickly find files for a date later

for f in nos_files:
    # Path structure: .../forecast/YYYYMMDD/taranto_nos_YYYYMMDD_nc4.nc
    try:
        date_str = f.split('/')[-2] # Parent folder is date
        set_cloud.add(date_str)
        
        # Store both NOS and assume OUS follows same pattern
        base_dir = os.path.dirname(f)
        nos_path = f's3://{f}'
        ous_path = f's3://{base_dir}/taranto_ous_{date_str}_nc4.nc'
        
        date_to_files_map[date_str] = {'nos': nos_path, 'ous': ous_path}
        
    except IndexError:
        pass

print(f"set_cloud: {len(set_cloud)} dates found.")

In [ ]:
# --- 3. Determine New Dates ---
new_dates = sorted(list(set_cloud - set_repo))

print(f"Dates to process: {len(new_dates)}")
if new_dates:
    print(f"Range: {new_dates[0]} to {new_dates[-1]}")

### Step 2: Virtualize and Merge New Data

In [ ]:
def fix_ds(ds):
    """Standardizes dimensions and coordinates for Taranto dataset."""
    ds = ds.rename_vars(time='valid_time')
    ds = ds.rename_dims(time='step')
    step = (ds.valid_time - ds.valid_time[0]).assign_attrs({"standard_name": "forecast_period"})
    time = ds.valid_time[0].assign_attrs({"standard_name": "forecast_reference_time"})
    ds = ds.assign_coords(step=step, time=time)
    ds = ds.drop_indexes("valid_time")
    ds = ds.drop_vars('valid_time')
    ds = ds.set_coords(['latitude', 'longitude', 'element_index', 'topology', 'total_depth'])
    return ds

In [ ]:
ds_final = None

# new_dates = new_dates[:3]  # only if testing

if new_dates:
    # Reconstruct file lists based on the identified dates
    nos_urls = [date_to_files_map[d]['nos'] for d in new_dates]
    ous_urls = [date_to_files_map[d]['ous'] for d in new_dates]

    # --- Process NOS ---
    print(f"Virtualizing {len(nos_urls)} NOS files...")
    nos_list = [
        open_virtual_dataset(url, parser=parser, registry=registry, loadable_variables=["time"])
        for url in nos_urls
    ]
    nos_list = [fix_ds(ds) for ds in nos_list]
    combined_nos = xr.concat(
        nos_list, dim="time", coords="minimal", compat="override", combine_attrs="override"
    )

    # --- Process OUS ---
    print(f"Virtualizing {len(ous_urls)} OUS files...")
    ous_list = [
        open_virtual_dataset(url, parser=parser, registry=registry, loadable_variables=["time"])
        for url in ous_urls
    ]
    ous_list = [fix_ds(ds) for ds in ous_list]
    combined_ous = xr.concat(
        ous_list, dim="time", coords="minimal", compat="override", combine_attrs="override"
    )

    # --- Merge ---
    ds_final = xr.merge([combined_nos, combined_ous], compat='override')
    print("Datasets merged and ready for writing to Icechunk.")
    
else:
    print("No new dates found. Skipping processing.")

### Step 3: Append to Icechunk

In [ ]:
if ds_final is not None:
    # Ensure we have a valid repo object
    if repo is None:
        repo = icechunk.Repository.create(storage, config, authorize_virtual_chunk_access=credentials)
        initial_session = repo.writable_session("main")

        # Append
        print(f"Writing {len(ds_final.time)} time steps to Icechunk...")
        ds_final.virtualize.to_icechunk(initial_session.store)
    
        # Commit
        msg = f"Initialized with forecast data: {new_dates[0]} to {new_dates[-1]}"
        initial_session.commit(msg)
        print(f"Commit successful: '{msg}'")
    # Create Writable Session
    else:
        append_session = repo.writable_session("main")

        # Append
        print(f"Appending {len(ds_final.time)} time steps to Icechunk...")
        ds_final.virtualize.to_icechunk(append_session.store, append_dim="time")
    
        # Commit
        msg = f"Appended forecast data: {new_dates[0]} to {new_dates[-1]}"
        append_session.commit(msg)
        print(f"Commit successful: '{msg}'")

    # Verify History
    history = repo.ancestry(branch="main")
    latest = next(history)
    print(f"Latest Commit [{latest.written_at}]: {latest.message}")
    
else:
    print("Nothing to append.")